# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [6]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [7]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [8]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [9]:
#Resumen estadístico de orders
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [10]:
#Resumen estadístico de catalog
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [11]:
#Resumen estadístico de marketing
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [12]:
# Orders duplicadas
print("Filas duplicadas:", orders.duplicated().sum())
orders[orders.duplicated(keep=False)].sort_values(by='id_pedido').head(10)

Filas duplicadas: 100


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55


In [13]:
#Filas duplicadas de catalog
print("Filas duplicadas:", catalog.duplicated().sum())

Filas duplicadas: 0


In [14]:
#Filas duplicadas de Marketing
print("Filas duplicadas:", marketing.duplicated().sum())

Filas duplicadas: 0


In [15]:
#Eliminar orders duplicadas
orders = orders.drop_duplicates(keep='first').reset_index(drop=True)
orders.duplicated().sum()

0

📝**Comentario:** Se identificaron y eliminaron 100 registros duplicados en la tabla orders, equivalentes al 0,4 % de las 25.100 filas originales. La revisión confirmó que se trataba de copias exactas de pedidos ya existentes, con el mismo id_pedido y valores idénticos en todas las columnas. Además, los registros duplicados se encontraban al final del conjunto de datos, lo que sugiere un posible error durante el proceso de carga o consolidación de la información, más que pedidos reales adicionales.

Mantener estos registros habría generado una sobreestimación del número de pedidos y de los ingresos (revenue), afectando la precisión de métricas clave como el embudo de conversión (funnel) y el análisis de cohortes. Por esta razón, se conservó únicamente la primera aparición de cada pedido y se eliminaron las duplicadas. Como resultado, el conjunto de datos quedó conformado por 25.000 registros únicos.

In [16]:
#Unificar las categorías en orders y catalog
print("Categoría Prod en Orders:",orders["categoria_producto"].unique())
print("Categoría Prod en Catalog:",catalog["categoria_producto"].unique())

Categoría Prod en Orders: ['Moda' 'Electronica' 'Hogar' nan]
Categoría Prod en Catalog: ['Electrónica' 'Hogar' 'Moda']


In [17]:
#Reemplazar las categorías en orders
orders["categoria_producto"]= orders["categoria_producto"].replace("Electronica","Electrónica")
orders["categoria_producto"].isna().sum()

80

In [18]:
#Se crea diccionario de producto 
mapa_categoria = catalog.set_index("nombre_producto")["categoria_producto"].to_dict()
orders["categoria_producto"] = orders["categoria_producto"].fillna(orders["nombre_producto"].map(mapa_categoria))
orders["categoria_producto"].isna().sum()

30

In [19]:
orders[orders["categoria_producto"].isna()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,433.76,0.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,495.38,5.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,382.94,0.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,415.01,0.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,236.49,0.0,472.98


📝 **Comentario:** Se estandarizó la nomenclatura de la categoría Electrónica en la tabla orders, corrigiendo los registros que aparecían como “Electronica” sin tilde. Esta normalización permitió unificar los valores con los registrados en el catálogo y evitar que una misma categoría se contabilizara como dos categorías diferentes durante el análisis.

Adicionalmente, se detectaron 80 valores nulos en la columna categoria_producto, equivalentes al 0,3 % del total de registros. Dado que cada producto está asociado a una única categoría en la tabla de catálogo, se construyó un diccionario de correspondencia producto → categoría (mapa_categoria) y se utilizó junto con .fillna() para completar únicamente los valores faltantes, preservando la información ya existente.

Mediante este procedimiento fue posible recuperar la categoría de 50 registros. Los 30 casos restantes no pudieron completarse debido a que tampoco contaban con información en la columna nombre_producto, lo que impedía establecer una correspondencia con el catálogo. El tratamiento aplicado a estos registros se describe en el siguiente apartado del proceso de limpieza de datos.

In [20]:
#Eliminar nulos en categoria_producto
orders = orders.dropna(subset=["categoria_producto"])
orders.shape

(24970, 12)

📝 **Comentario:** Se eliminaron las 30 filas restantes que presentaban valores nulos tanto en nombre_producto como en categoria_producto, lo que representa aproximadamente el 0,12 % del conjunto de datos. Tras esta depuración, la tabla quedó conformada por 24.970 registros.

La decisión se tomó debido a que, al no contar con el nombre del producto, no era posible identificar su categoría ni recuperar esta información a partir del catálogo. En consecuencia, estos registros tampoco permitían calcular correctamente los costos asociados, una variable indispensable para el análisis de rentabilidad.

Mantener estas filas habría introducido información incompleta en las etapas posteriores del análisis, con el riesgo de distorsionar los resultados y sesgar la interpretación de métricas clave, especialmente en la identificación y visualización de los productos con mayor contribución al monto total de ventas.

In [21]:
#Validación de nulos eliminados
print("Categorías únicas en Orders:",orders["categoria_producto"].unique())
print("Categorías únicas en Catalog:",catalog["categoria_producto"].unique())

Categorías únicas en Orders: ['Moda' 'Electrónica' 'Hogar']
Categorías únicas en Catalog: ['Electrónica' 'Hogar' 'Moda']


In [22]:
columnas_numericas = [
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]

# Contar valores negativos
print("Valores negativos:")
print((orders[columnas_numericas] < 0).sum())

# Contar valores iguales a cero
print("Valores iguales a cero:")
print((orders[columnas_numericas] == 0).sum())

Valores negativos:
cantidad           4
precio_unitario    0
monto_descuento    0
monto_total        4
dtype: int64
Valores iguales a cero:
cantidad               0
precio_unitario        0
monto_descuento    12481
monto_total            0
dtype: int64


In [23]:
#Identificar registros negativos
registros_negativos = orders[
    (orders[columnas_numericas] < 0).any(axis=1)
]

display(registros_negativos)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electrónica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electrónica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-1.0,423.53,0.0,-423.53


In [24]:
#Identificar resgistros en cero
registros_cero = orders[
    (orders[columnas_numericas] == 0).any(axis=1)
]

display(registros_cero)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
5,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electrónica,1.0,179.34,0.0,179.34
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.19
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electrónica,2.0,373.68,0.0,747.35
...,...,...,...,...,...,...,...,...,...,...,...,...
24991,order_24991,user_5504,2025-05-06,mexico,desktop,social,Jacket-Winter-M,Moda,2.0,98.63,0.0,197.26
24992,order_24992,user_2301,2025-03-23,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,212.40,0.0,212.40
24995,order_24995,user_4217,2025-04-16,Argentina,mobile,social,Tablet-Standard-64GB,Electrónica,1.0,64.57,0.0,64.57
24997,order_24997,user_3119,2025-03-27,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,226.85,0.0,226.85


In [25]:
orders = orders[~(orders["cantidad"] < 0)]
print("Valores Negativos en columna Cantidad:", (orders["cantidad"] < 0).sum())

Valores Negativos en columna Cantidad: 0


📝 **Comentario:** Se identificaron 4 registros con valores negativos en las columnas cantidad y monto_total, lo que representa aproximadamente el 0,02 % del conjunto de datos. Dado que no se dispone de una definición clara por parte del área responsable sobre el significado de estos registros, se decidió excluirlos temporalmente del análisis para preservar la consistencia de la información.

Mantener estos valores podría afectar los resultados del análisis de rentabilidad, ya que reducirían el revenue total sin tener certeza de si corresponden a devoluciones, ajustes contables u otro tipo de operación legítima. Ante esta incertidumbre, se optó por trabajar únicamente con registros de ventas positivas, minimizando así el riesgo de sesgos en las métricas y conclusiones del análisis.

Como resultado de esta depuración, el conjunto de datos quedó compuesto por 24.966 registros.

Pendiente: validar con el área responsable del negocio si estos valores negativos representan transacciones válidas (por ejemplo, devoluciones o notas de crédito) y determinar si deben reincorporarse en futuros análisis.

In [26]:
#Filas y columnas actualizadas
print("Filas y columnas actualizadas:", orders.shape)
orders.describe()

Filas y columnas actualizadas: (24966, 12)


,cantidad,precio_unitario,monto_descuento,monto_total
count,24916.000000,24916.000000,24916.000000,2.496600e+04
mean,7.123214,259.367124,4.502930,2.081885e+03
std,297.072369,138.670945,5.224131,9.921507e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.535000,0.000000,1.807575e+02
50%,2.000000,258.775000,0.000000,3.419250e+02
75%,2.000000,380.355000,10.000000,5.185475e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [27]:
#Revisar distribución de valores en la columna cantidad
orders["cantidad"].value_counts().sort_index()

1.0        12330
2.0        12576
10000.0        6
20000.0        4
Name: cantidad, dtype: int64

In [28]:
#Valores inusuales
orders[orders["cantidad"] > 2]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,43.14,0.0,431400.0
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electrónica,10000.0,280.55,0.0,2805500.0
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,490.35,0.0,4903500.0
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electrónica,10000.0,238.15,0.0,2381500.0
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electrónica,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,348.31,0.0,6966200.0
3689,order_3689,user_6566,2025-06-16,mexico,desktop,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,87.69,0.0,876900.0
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electrónica,20000.0,290.85,0.0,5817000.0
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electrónica,10000.0,336.93,0.0,3369300.0


In [29]:
# Eliminar valores inusuales
orders = orders[~(orders["cantidad"]>2)]
orders.shape

(24956, 12)

📝 **Comentario:** Durante la revisión estadística de la variable cantidad, se identificaron 10 registros atípicos, equivalentes al 0,04 % del conjunto de datos. Estos registros presentaban volúmenes de compra excepcionalmente altos, entre 10.000 y 20.000 unidades en una sola transacción, correspondientes al producto Laptop-Gaming-16GB. En consecuencia, los valores de monto_total alcanzaban cifras de hasta 8,8 millones, muy por encima de los niveles observados en el resto de los datos, donde los valores máximos habituales se situaban alrededor de las mil unidades.

Dado que estas observaciones se encontraban muy alejadas del comportamiento general de la muestra y no se cuenta con validación por parte del área responsable sobre si corresponden a compras reales o a posibles errores de captura, se optó por excluirlas del análisis. Mantenerlas habría generado una influencia desproporcionada en las visualizaciones y en el cálculo del revenue, afectando la representatividad de los resultados.

Asimismo, se descartó la aplicación de técnicas de winsorización, ya que ello implicaría reemplazar los valores extremos por cantidades arbitrarias que no están respaldadas por la información disponible.

Como resultado de esta depuración, el conjunto de datos quedó conformado por 24.956 registros.

Pendiente: confirmar con el área responsable si estos valores corresponden a transacciones válidas o a errores en el registro de la información, con el fin de evaluar su posible reincorporación en futuros análisis.

In [30]:
#Resumen estadístico orders
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,24906.000000,24906.000000,24906.000000,24956.000000
mean,1.504939,259.356605,4.504738,385.945141
std,0.499986,138.672693,5.224400,255.643812
min,1.000000,20.030000,0.000000,5.240000
25%,1.000000,138.525000,0.000000,180.670000
50%,2.000000,258.735000,0.000000,341.710000
75%,2.000000,380.365000,10.000000,517.680000
max,2.000000,499.960000,15.000000,999.890000


In [31]:
#Monto total calculado
orders["monto_calculado"] = orders["cantidad"] * orders["precio_unitario"] - orders["monto_descuento"]
print(orders.head())

  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electrónica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electrónica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  monto_calculado  
0           332.69              0.0       665.37           665.38  
1           176.

In [32]:
#Diferencia entre el monto calculado y la columna monto total
orders["diferencia"] = (orders["monto_total"] - orders["monto_calculado"]).abs()
print(orders["diferencia"].describe())

count    2.490600e+04
mean     2.498996e-03
std      4.329634e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      1.000000e-02
Name: diferencia, dtype: float64


In [33]:
#Filtrar nulos en monto calculado
orders[orders["monto_calculado"].isna()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado,diferencia
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,595.85,NaN,NaN
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,458.15,NaN,NaN
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,NaN,NaN,NaN,319.75,NaN,NaN
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,227.55,NaN,NaN
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,Electrónica,NaN,NaN,NaN,432.39,NaN,NaN
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,Moda,NaN,NaN,NaN,527.15,NaN,NaN
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,711.18,NaN,NaN
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,Electrónica,NaN,NaN,NaN,41.08,NaN,NaN
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,Moda,NaN,NaN,NaN,431.55,NaN,NaN
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,279.35,NaN,NaN


In [34]:
#Eliminar valores nulos en cantidad
orders = orders.dropna(subset=["cantidad"])
print(orders.shape)

(24906, 14)


In [35]:
#Eliminar columna monto calculado y diferencia
orders = orders.drop(["monto_calculado","diferencia"],axis=1)
print(orders.shape)

(24906, 12)


📝 **Comentario:** Se verificó la consistencia de la columna monto_total en todos los registros con información completa, comprobando que su valor correspondiera al cálculo esperado: cantidad × precio_unitario − descuento. La validación evidenció únicamente diferencias mínimas atribuibles al redondeo de decimales, con una desviación máxima de 0,01, por lo que se concluyó que no existían inconsistencias significativas en los cálculos.

Por otra parte, se identificaron 50 registros que presentaban valores nulos simultáneamente en las columnas cantidad, precio_unitario y descuento. Aunque estos registros contaban con información en monto_total, no era posible reconstruir de forma fiable los valores faltantes, ya que existen múltiples combinaciones posibles de cantidad, precio y descuento que podrían generar el mismo importe final. Además, la ausencia de la variable cantidad impedía calcular los costos asociados, una métrica necesaria para el análisis de rentabilidad.

Dado que estos registros representaban únicamente el 0,20 % del conjunto de datos y que su conservación podía introducir incertidumbre en las etapas posteriores del análisis, se optó por eliminarlos. Como resultado, el conjunto de datos quedó conformado por 24.906 registros.

Finalmente, se eliminaron las columnas auxiliares utilizadas durante el proceso de validación, manteniendo únicamente las variables relevantes para el análisis.

In [36]:
#Contar valores nulos en orders
orders.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia       0
nombre_producto         0
categoria_producto      0
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64

In [37]:
#Agrupar paises por usuario
paises_por_usuario = orders.groupby("id_usuario")["pais"].nunique()
print((paises_por_usuario > 1).sum())

1997


In [38]:
#paises unicos en orders y marketing
print(orders["pais"].unique())
print(marketing["pais"].unique())

['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' nan 'argentina']
['Mexico' 'Colombia' 'Argentina']


In [39]:
#Limpiar y estantarizar nombre de paises en la columnna pais
orders["pais"] = orders["pais"].str.strip().str.title()

In [40]:
#valores unicos
orders["pais"].unique()

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

In [41]:
#Reemplazar valores nulos por "Desconocido"
orders["pais"] = orders["pais"].fillna("Desconocido")

In [42]:
#Contar paises
orders["pais"].value_counts()

Mexico         8308
Colombia       8280
Argentina      8022
Desconocido     296
Name: pais, dtype: int64

In [43]:
#Valores unicos en paises en orders y marketing
print(orders["pais"].unique())
print(marketing["pais"].unique())

['Argentina' 'Mexico' 'Colombia' 'Desconocido']
['Mexico' 'Colombia' 'Argentina']


📝 **Comentario:** Se estandarizó el formato de los nombres de los tres países en las tablas orders y marketing para garantizar la consistencia de la información y facilitar los cruces entre ambas fuentes.

Posteriormente, se identificaron 296 valores nulos en la columna pais de la tabla orders, equivalentes al 1,19 % de los 24.906 registros. Con el objetivo de recuperar esta información, se evaluó la posibilidad de asignar el país a partir del historial de compras de cada usuario. Sin embargo, el análisis mostró que 1.997 de los 7.640 usuarios únicos (26,14 %) habían realizado compras desde más de un país, lo que impedía determinar de forma fiable un único país de referencia para completar los valores faltantes.

Dado que los registros con país faltante representan una proporción reducida del conjunto de datos y que el resto de sus variables se encontraban completas, se decidió conservarlos. Para facilitar su identificación y seguimiento en las etapas posteriores del análisis, los valores nulos fueron reemplazados por la categoría “Desconocido”.

Es importante tener en cuenta que en la tabla marketing no existe información de inversión asociada a la categoría “Desconocido”. Por ello, esta situación deberá considerarse al interpretar los resultados del análisis de rentabilidad por país, ya que dichos registros no podrán vincularse directamente con un gasto de marketing específico.

In [44]:
#Nulos en dispositivo en orders
print("Nulos en columna Dispositivo:", orders["dispositivo"].isna().sum())
orders[orders["dispositivo"].isna()]

Nulos en columna Dispositivo: 20


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99
25,order_25,user_540,2025-03-08,Argentina,NaN,social,Vacuum-Pro-Black,Hogar,1.0,183.50,5.0,178.50
26,order_26,user_1951,2025-02-10,Colombia,NaN,organic,Blender-XL-Red,Hogar,1.0,331.52,15.0,316.52
27,order_27,user_5410,2025-05-09,Colombia,NaN,paid_search,Phone-Pro-128GB,Electrónica,2.0,449.48,5.0,893.96
28,order_28,user_1489,2025-01-17,Mexico,NaN,social,Phone-Pro-128GB,Electrónica,1.0,60.70,10.0,50.70
29,order_29,user_5197,2025-01-30,Colombia,NaN,social,Phone-Pro-128GB,Electrónica,1.0,381.57,0.0,381.57
30,order_30,user_5386,2025-01-31,Mexico,NaN,social,Vacuum-Pro-Black,Hogar,1.0,183.74,10.0,173.74
31,order_31,user_5766,2025-05-05,Colombia,NaN,paid_search,Vacuum-Pro-Black,Hogar,1.0,480.74,0.0,480.74
32,order_32,user_4400,2025-03-29,Mexico,NaN,social,Tablet-Standard-64GB,Electrónica,1.0,117.69,0.0,117.69
33,order_33,user_5070,2025-01-12,Argentina,NaN,paid_search,Tablet-Standard-64GB,Electrónica,1.0,96.82,10.0,86.82


In [45]:
#Valores unicos id_usuario en orders
orders["id_usuario"].nunique()

7640

In [46]:
#Reemplazar nulos en dispositivo por "Desconocido"
orders["dispositivo"] = orders["dispositivo"].fillna("Desconocido")
print(orders["dispositivo"].value_counts())

desktop        12661
mobile         12225
Desconocido       20
Name: dispositivo, dtype: int64


📝 **Comentario:** Se identificaron 20 valores nulos en la columna dispositivo, equivalentes al 0,08 % del total de registros. Dado que el resto de la información asociada a estos pedidos se encontraba completa y que no existían variables suficientes para inferir de manera confiable el dispositivo utilizado, se decidió conservar dichos registros en el conjunto de datos.

Para facilitar su identificación y evitar su pérdida durante el análisis, los valores faltantes fueron reemplazados por la categoría “Desconocido”. Esta decisión permite diferenciar claramente estos casos de las categorías “Desktop” y “Mobile”, garantizando la transparencia en el tratamiento de los datos faltantes.

El impacto de esta modificación sobre los resultados es mínimo, ya que únicamente incorpora una categoría adicional en los análisis y visualizaciones segmentados por dispositivo, preservando al mismo tiempo la integridad de los registros y el volumen total de pedidos analizados.

In [47]:
# Convertir fecha al formato correcto
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
print(orders["fecha_hora_pedido"].dtype)
print(marketing["fecha"].dtype)

datetime64[ns]
datetime64[ns]


In [48]:
#Convertit cantidad a entero
orders["cantidad"] = orders["cantidad"].astype("int64")
print(orders["cantidad"].dtypes)

int64


📝 **Comentario:** Se convirtieron las columnas que contienen información de fechas a formato datetime64 mediante la función pd.to_datetime(): fecha_hora_pedido en la tabla Orders y fecha en la tabla Marketing. Esta transformación fue necesaria porque ambas columnas se encontraban almacenadas como texto, lo que limitaba su uso en análisis temporales. Al convertirlas al tipo de dato adecuado, es posible ordenar cronológicamente los registros, agrupar la información por períodos de tiempo y realizar cruces consistentes entre ambas tablas.

Adicionalmente, la columna cantidad de la tabla Orders se convirtió de float64 a int64. Originalmente, esta variable se encontraba en formato decimal debido a la presencia de valores nulos, ya tratados durante el proceso de limpieza. Una vez eliminados dichos valores, fue posible realizar la conversión a número entero, que es el formato más apropiado para esta variable, ya que representa unidades vendidas y, por definición, estas deben expresarse como cantidades enteras.

In [49]:
# Contar cuántas veces aparece cada canal de marketing
marketing["canal"].value_counts()

paid_search    507
social         506
organic        506
Name: canal, dtype: int64

In [50]:
#Valores nulos en canal para marketing
marketing[marketing["canal"].isna()]

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
...,...,...,...,...,...
194,2025-01-22,Colombia,social_Colombia,NaN,772.46
195,2025-01-22,Argentina,organic_Argentina,NaN,1562.61
196,2025-01-22,Argentina,paid_search_Argentina,NaN,644.34
197,2025-01-22,Argentina,social_Argentina,NaN,2544.66


In [51]:
# Crear un identificador de prueba uniendo el canal de marketing y el país
marketing["canal_prueba"] = (marketing["canal"] + "_" + marketing["pais"])

# Visualizar las primeras filas para verificar el resultado
print(marketing.head())

# Contar cuántos registros tienen un identificador diferente al esperado
print((marketing["canal_prueba"] != marketing["id_campaña"]).sum())

       fecha      pais            id_campaña        canal    gasto  \
0 2025-01-01    Mexico        organic_Mexico      organic  2446.25   
1 2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34   
2 2025-01-01    Mexico         social_Mexico       social  2045.01   
3 2025-01-01  Colombia      organic_Colombia      organic  2597.21   
4 2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40   

           canal_prueba  
0        organic_Mexico  
1    paid_search_Mexico  
2         social_Mexico  
3      organic_Colombia  
4  paid_search_Colombia  
101


In [52]:
# Extraer el canal de marketing del identificador de campaña y contar su frecuencia
marketing["id_campaña"].str.rsplit("_", n=1).str[0].value_counts()

social         540
organic        540
paid_search    540
Name: id_campaña, dtype: int64

In [53]:
# Extraer el canal de marketing desde el identificador de campaña y verificar que no existan valores nulos
marketing["canal"] = marketing["id_campaña"].str.rsplit("_", n=1).str[0]
print(marketing["canal"].isna().sum())

0


In [54]:
# Eliminar la columna temporal utilizada para validar los identificadores de campaña
marketing = marketing.drop(["canal_prueba"], axis=1)

# Mostrar las columnas restantes del DataFrame
print(marketing.columns)

Index(['fecha', 'pais', 'id_campaña', 'canal', 'gasto'], dtype='object')


In [55]:
# Contar el número de registros duplicados en cada conjunto de datos
print("Duplicados en marketing:", marketing.duplicated().sum())
print("Duplicados en catalog:", catalog.duplicated().sum())

Duplicados en marketing: 0
Duplicados en catalog: 0


In [56]:
# Contar cuántos registros tienen la misma fecha, país y canal
print(marketing.duplicated(subset=["fecha", "pais", "canal"]).sum())

# Mostrar ejemplos de los registros duplicados para su revisión
print(
    marketing[
        marketing.duplicated(
            subset=["fecha", "pais", "canal"],
            keep=False
        )
    ]
    .sort_values(by=["fecha", "pais", "canal"])
    .head(10)
)

0
Empty DataFrame
Columns: [fecha, pais, id_campaña, canal, gasto]
Index: []


In [57]:
# Mostrar los valores únicos de la fuente de referencia en pedidos y del canal en marketing para identificar diferencias de nomenclatura
print(orders["fuente_referencia"].unique())
print(marketing["canal"].unique())

['organic' 'paid_search' 'social']
['organic' 'paid_search' 'social']


📝**Comentario:** Se identificaron 101 valores nulos en la columna canal de la tabla marketing, equivalentes al 6,2 % de los 1.620 registros. Con el fin de evaluar la posibilidad de recuperar esta información, se comparó el contenido de dicha columna con el campo id_campaña. Para ello, se creó una columna auxiliar denominada canal_prueba, construida mediante la concatenación de los campos canal, un guion bajo (_) y pais.

La validación confirmó que, en las 1.519 filas que contaban con información en canal, el valor generado coincidía exactamente con id_campaña. Las únicas discrepancias observadas correspondían a los 101 registros con valores nulos, lo que permitió concluir que la información faltante podía recuperarse de forma fiable a partir de dicho identificador.

Para completar los datos, se extrajo de id_campaña la porción de texto ubicada antes del último guion bajo, considerando que algunos nombres de canal, como paid_search, contienen guiones bajos en su estructura. Tras aplicar esta transformación, se recuperó la totalidad de los registros faltantes, dejando la columna canal sin valores nulos.

Posteriormente, se verificó que la variable contiene únicamente tres canales válidos y que no presenta inconsistencias de nomenclatura: organic, paid_search y social. Finalmente, se eliminó la columna auxiliar canal_prueba y se comprobó que no existen registros duplicados para la combinación de fecha, país y canal, garantizando la consistencia y calidad de la información para las etapas posteriores del análisis.

In [58]:
# Mostrar y ordenar los nombres únicos de los productos en ambas tablas para identificar diferencias de nomenclatura
print(sorted(orders["nombre_producto"].unique()))
print(sorted(catalog["nombre_producto"].unique()))

['Blender-XL-Red', 'Jacket-Winter-M', 'Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Sneakers-Urban-42', 'Tablet-Standard-64GB', 'Vacuum-Pro-Black']
['Blender-XL-Red', 'Jacket-Winter-M', 'Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Sneakers-Urban-42', 'Tablet-Standard-64GB', 'Vacuum-Pro-Black']


💡**Validaciones realizadas (sin requerir modificaciones)**
1. Consistencia del cálculo de monto_total: se verificó que el valor de monto_total coincide con el resultado esperado de la fórmula cantidad × precio_unitario − monto_descuento en todos los registros que cuentan con estas tres variables. Las únicas diferencias observadas fueron mínimas (hasta 0,01), atribuibles al redondeo de decimales.

2. Ausencia de registros duplicados: se confirmó que las tablas Marketing y Catalog no presentan registros duplicados. Adicionalmente, en la tabla Marketing se verificó que tampoco existen duplicados para la combinación de fecha, país y canal.

3. Consistencia entre productos de Orders y Catalog: se comprobó que los valores de nombre_producto son consistentes entre las tablas Orders y Catalog, encontrándose los mismos 7 productos en ambas fuentes. Esto garantiza que todas las ventas pueden asociarse correctamente a un costo dentro del análisis de rentabilidad.

4. Integridad de la información para el análisis: tras las actividades de limpieza y validación realizadas, los datos presentan un nivel adecuado de consistencia y calidad para continuar con las etapas de integración, análisis exploratorio y evaluación de rentabilidad.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [59]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [60]:
# Unir Orders con Catalog
orders_cost = orders.merge(
    catalog,
    on="nombre_producto",
    how="left"
)

print(orders_cost.head())


  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto_x  cantidad  \
0           organic       Jacket-Winter-M                 Moda         2   
1       paid_search  Tablet-Standard-64GB          Electrónica         1   
2            social        Blender-XL-Red                Hogar         2   
3            social  Tablet-Standard-64GB          Electrónica         1   
4       paid_search        Blender-XL-Red                Hogar         1   

   precio_unitario  monto_descuento  monto_total categoria_producto_y  \
0           332.69              0.0       665.37                 Mo

In [61]:
#Revenue total

revenue_total = orders_cost["monto_total"].sum()

print(f"Revenue total: ${revenue_total:,.2f}")

Revenue total: $9,610,018.94


In [62]:
#Costo total

orders_cost["costo_total"] = (
    orders_cost["cantidad"] *
    orders_cost["costo_unitario"]
)

costo_total = orders_cost["costo_total"].sum()

print(f"Costo total: ${costo_total:,.2f}")

Costo total: $3,828,869.01


In [63]:
#Inversión total en marketing

inversion_marketing = marketing["gasto"].sum()

print(f"Inversión total en marketing: ${inversion_marketing:,.2f}")

Inversión total en marketing: $2,871,843.53


In [64]:
#Profit (beneficio)

profit = (
    revenue_total
    - costo_total
    - inversion_marketing
)
margen_profit = profit / revenue_total * 100

print(f"Profit: ${profit:,.2f}")
print(f"margen_profit: {margen_profit:,.2f}%")

Profit: $2,909,306.40
margen_profit: 30.27%


In [65]:
#¿El negocio es rentable?
if profit > 0:
    print("✅ El negocio es rentable")
else:
    print("❌ El negocio no es rentable")

✅ El negocio es rentable


📝**Comentario:** Para calcular los KPIs principales del negocio, se integró la información de ventas de Orders con los costos unitarios contenidos en Catalog, permitiendo asociar a cada transacción su costo correspondiente. A partir de esta integración, se calculó el revenue total mediante la suma de los montos de venta, el costo total a partir de las cantidades vendidas y los costos unitarios, y la inversión total en marketing utilizando los datos de la tabla Marketing. Posteriormente, se estimó el profit como la diferencia entre los ingresos generados y los costos totales del negocio, incluyendo la inversión en marketing. Finalmente, se evaluó la rentabilidad general de la operación con base en este resultado, determinando si los ingresos obtenidos fueron suficientes para cubrir los costos de venta y adquisición de clientes.

In [66]:
#Calcular la utilidad por venta
orders_cost["profit"] = (
    orders_cost["monto_total"] -
    orders_cost["costo_total"]
)

In [67]:
#Filtrar las ventas con pérdida
ventas_perdida = orders_cost[
    orders_cost["profit"] < 0
]

print(ventas_perdida.head())

   id_pedido id_usuario fecha_hora_pedido       pais  dispositivo  \
2    order_2  user_3194        2025-05-02  Argentina      desktop   
6    order_6  user_2802        2025-03-15   Colombia       mobile   
19  order_19  user_7190        2025-02-03  Argentina       mobile   
39  order_39  user_7855        2025-01-29   Colombia  Desconocido   
41  order_41  user_2963        2025-01-27     Mexico  Desconocido   

   fuente_referencia  nombre_producto categoria_producto_x  cantidad  \
2             social   Blender-XL-Red                Hogar         2   
6            organic   Blender-XL-Red                Hogar         2   
19            social  Jacket-Winter-M                 Moda         2   
39       paid_search   Blender-XL-Red                Hogar         1   
41            social  Jacket-Winter-M                 Moda         2   

    precio_unitario  monto_descuento  monto_total categoria_producto_y  \
2            102.99             10.0       195.99                Hogar   
6   

In [68]:
#Contar cuántas ventas generan pérdidas
print("Ventas con pérdida:", len(ventas_perdida))

Ventas con pérdida: 4528


In [69]:
#% de ventas con perdidas de producto
ventas_perdida["nombre_producto"].value_counts()

Jacket-Winter-M         1525
Laptop-Gaming-16GB      1498
Blender-XL-Red          1414
Tablet-Standard-64GB      48
Sneakers-Urban-42         31
Vacuum-Pro-Black          10
Phone-Pro-128GB            2
Name: nombre_producto, dtype: int64

In [70]:
#Ver cuánto dinero se pierde en total
perdida_total = ventas_perdida["profit"].sum()

print(f"Pérdida total: ${perdida_total:,.2f}")

Pérdida total: $-665,567.82


In [71]:
#Ver qué productos generan más pérdidas
ventas_perdida.groupby("nombre_producto")["profit"] \
              .sum() \
              .sort_values()

nombre_producto
Laptop-Gaming-16GB     -305320.23
Jacket-Winter-M        -195120.14
Blender-XL-Red         -164571.92
Tablet-Standard-64GB      -385.23
Sneakers-Urban-42         -118.99
Vacuum-Pro-Black           -44.99
Phone-Pro-128GB             -6.32
Name: profit, dtype: float64

In [72]:
# Total de ventas por producto
total_ventas = orders_cost["nombre_producto"].value_counts()

# Ventas con pérdida por producto
ventas_perdida_prod = ventas_perdida["nombre_producto"].value_counts()

# Porcentaje de ventas con pérdida
porcentaje_perdida = (
    ventas_perdida_prod / total_ventas * 100
).sort_values(ascending=False)

print(porcentaje_perdida)

Laptop-Gaming-16GB      54.118497
Jacket-Winter-M         36.605857
Blender-XL-Red          33.860153
Tablet-Standard-64GB     1.736614
Sneakers-Urban-42        0.750787
Vacuum-Pro-Black         0.239808
Phone-Pro-128GB          0.073180
Name: nombre_producto, dtype: float64


In [73]:
#rentabilidad del producto
rentabilidad_producto = (
    orders_cost
    .groupby("nombre_producto")
    .agg(
        revenue=("monto_total", "sum"),
        costo=("costo_total", "sum"),
        profit=("profit", "sum")
    )
    .sort_values("profit")
)

print(rentabilidad_producto)

                         revenue       costo      profit
nombre_producto                                         
Laptop-Gaming-16GB    1084672.98  1178294.64   -93621.66
Jacket-Winter-M       1606586.17  1184323.36   422262.81
Blender-XL-Red        1611574.59  1109122.56   502452.03
Tablet-Standard-64GB  1062922.80   104697.13   958225.67
Phone-Pro-128GB       1057134.62    41896.80  1015237.82
Sneakers-Urban-42     1566075.93   106220.12  1459855.81
Vacuum-Pro-Black      1621051.85   104314.40  1516737.45


In [74]:
#Magen rentabilidad del producto
rentabilidad_producto["margen_%"] = (
    rentabilidad_producto["profit"]
    / rentabilidad_producto["revenue"]
) * 100

print(rentabilidad_producto)

                         revenue       costo      profit   margen_%
nombre_producto                                                    
Laptop-Gaming-16GB    1084672.98  1178294.64   -93621.66  -8.631326
Jacket-Winter-M       1606586.17  1184323.36   422262.81  26.283234
Blender-XL-Red        1611574.59  1109122.56   502452.03  31.177709
Tablet-Standard-64GB  1062922.80   104697.13   958225.67  90.150072
Phone-Pro-128GB       1057134.62    41896.80  1015237.82  96.036758
Sneakers-Urban-42     1566075.93   106220.12  1459855.81  93.217435
Vacuum-Pro-Black      1621051.85   104314.40  1516737.45  93.565018


In [75]:
#Revenue por canal/fuente de referencia
revenue_canal = (
    orders.groupby("fuente_referencia")["monto_total"]
    .sum()
    .reset_index()
)

print(revenue_canal)

  fuente_referencia  monto_total
0           organic   3178228.55
1       paid_search   3209280.07
2            social   3222510.32


In [76]:
#Gasto total por canal 
gasto_canal = (
    marketing.groupby("canal")["gasto"]
    .sum()
    .reset_index()
)

print(gasto_canal)

         canal      gasto
0      organic  972650.96
1  paid_search  922374.20
2       social  976818.37


In [77]:
#Unir ambas tablas
eficiencia_canal = revenue_canal.merge(
    gasto_canal,
    left_on="fuente_referencia",
    right_on="canal",
    how="inner"
)

eficiencia_canal["roas"] = (
    eficiencia_canal["monto_total"]
    / eficiencia_canal["gasto"]
)

print(eficiencia_canal.sort_values("roas", ascending=False))

  fuente_referencia  monto_total        canal      gasto      roas
1       paid_search   3209280.07  paid_search  922374.20  3.479369
2            social   3222510.32       social  976818.37  3.298986
0           organic   3178228.55      organic  972650.96  3.267594


In [78]:
#Calcular ROAS
eficiencia_canal["roas"] = (
    eficiencia_canal["monto_total"] /
    eficiencia_canal["gasto"]
)

eficiencia_canal.sort_values(
    by="roas",
    ascending=False
)

,fuente_referencia,monto_total,canal,gasto,roas
1,paid_search,3209280.07,paid_search,922374.20,3.479369
2,social,3222510.32,social,976818.37,3.298986
0,organic,3178228.55,organic,972650.96,3.267594


📝**Comentario:** Se evaluó la rentabilidad del negocio desde dos perspectivas: la utilidad generada por las ventas y la eficiencia de los canales de adquisición. En primer lugar, se identificaron 4.528 ventas con utilidad negativa, equivalentes al 18,18 % de las 24.906 transacciones analizadas, lo que representa una pérdida acumulada de $665.567,82. Estas operaciones se concentraron principalmente en Jacket-Winter-M (1.525 ventas con pérdida), Laptop-Gaming-16GB (1.498) y Blender-XL-Red (1.414).

Sin embargo, al consolidar los resultados por producto, se observó que Laptop-Gaming-16GB es el único artículo que presenta una rentabilidad neta negativa. Este producto generó ingresos por 1.084.672,98, frente a costos por 1.178.294,64, produciendo una pérdida acumulada de 93.621,66 y un margen de -8,63 %. Además, el 54,12 % de sus ventas fueron realizadas con pérdida. Por el contrario, Jacket-Winter-M y Blender-XL-Red mantienen beneficios acumulados de 422.262,81 y 502.452,03, respectivamente, a pesar de registrar un número importante de transacciones no rentables.

Adicionalmente, se analizó la eficiencia de los canales de adquisición mediante el indicador ROAS (Return On Advertising Spend). Los resultados muestran que los tres canales generan retornos positivos sobre la inversión en marketing. Paid Search obtuvo el mejor desempeño, con un ROAS de 3,48, seguido de Social con 3,30 y Organic con 3,27. Esto significa que, por cada 1 invertido, estos canales generan aproximadamente 3,48, 3,30 y 3,27 en ingresos, respectivamente.

💡**Hallazgo clave**
Aunque los canales de marketing presentan niveles de eficiencia similares y generan retornos positivos, la principal oportunidad de mejora en la rentabilidad del negocio se encuentra en Laptop-Gaming-16GB, ya que es el único producto con pérdidas acumuladas y concentra más de la mitad de sus ventas en transacciones deficitarias. Por ello, se recomienda revisar su estructura de costos, política de descuentos y estrategia de precios antes de aumentar la inversión en adquisición de tráfico.

In [79]:
#Ticket promedio por orden
ticket_promedio = orders["monto_total"].mean()

print(f"Ticket promedio: ${ticket_promedio:,.2f}")

Ticket promedio: $385.85


In [80]:
#Cantidad promedio de productos por orden
cantidad_promedio = (
    orders.groupby("id_pedido")["cantidad"]
    .sum()
    .mean()
)

print(f"Cantidad promedio por orden: {cantidad_promedio:.2f}")


Cantidad promedio por orden: 1.50


In [81]:
#Producto más vendido
producto_mas_vendido = (
    orders.groupby("nombre_producto")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)

print(producto_mas_vendido)

nombre_producto
Vacuum-Pro-Black        6284
Blender-XL-Red          6279
Jacket-Winter-M         6256
Sneakers-Urban-42       6172
Laptop-Gaming-16GB      4198
Tablet-Standard-64GB    4153
Phone-Pro-128GB         4140
Name: cantidad, dtype: int64


In [82]:
print("Producto más vendido:")
print(producto_mas_vendido.head(1))

Producto más vendido:
nombre_producto
Vacuum-Pro-Black    6284
Name: cantidad, dtype: int64


In [83]:
#Gasto en marketing por canal
gasto_por_canal = (
    marketing.groupby("canal")["gasto"]
    .sum()
    .sort_values(ascending=False)
)

print(gasto_por_canal)

canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


📝**Comentario:** Se analizaron los principales indicadores relacionados con el comportamiento de las ventas para comprender los patrones de compra de los clientes y el desempeño comercial del negocio. Para ello, se calculó el ticket promedio por pedido, la cantidad promedio de productos por orden, el producto más vendido y la distribución de la inversión en marketing por canal. Estos indicadores permiten identificar cómo compran los clientes, cuáles son los productos con mayor demanda y cómo se distribuyen los esfuerzos de adquisición entre los distintos canales de marketing. En conjunto, este análisis complementa la evaluación de rentabilidad al proporcionar contexto sobre el volumen de ventas, la composición de los pedidos y la contribución de cada canal a la generación de ingresos.

In [84]:
#🎯 Resumen de KPIs
print("===== KPIs DEL NEGOCIO =====")

print(f"Revenue Total: ${revenue_total:,.2f}")
print(f"Costo Total: ${costo_total:,.2f}")
print(f"Inversión Marketing: ${inversion_marketing:,.2f}")
print(f"Profit: ${profit:,.2f}")
print(f"margen_profit: {margen_profit:,.2f}%")
print(f"Ticket Promedio: ${ticket_promedio:,.2f}")
print(f"Cantidad Promedio por Orden: {cantidad_promedio:.2f}")
print(f"Producto Más Vendido: {producto_mas_vendido.index[0]}")

===== KPIs DEL NEGOCIO =====
Revenue Total: $9,610,018.94
Costo Total: $3,828,869.01
Inversión Marketing: $2,871,843.53
Profit: $2,909,306.40
margen_profit: 30.27%
Ticket Promedio: $385.85
Cantidad Promedio por Orden: 1.50
Producto Más Vendido: Vacuum-Pro-Black


💡**Conclusión - Comportamiento de ventas y rentabilidad**
El negocio presenta un desempeño financiero positivo, con un revenue total de 9.610.018,94, un profit de 2.909.306,40 y un margen de rentabilidad del 30,27 %, lo que indica que los ingresos generados son suficientes para cubrir tanto los costos de producto como la inversión en marketing. Además, el ticket promedio por pedido fue de $385,85, mientras que la cantidad promedio de productos por orden fue de 1,5 unidades, lo que sugiere que la mayoría de las compras corresponden a pedidos pequeños de uno o dos artículos. Asimismo, Vacuum-Pro-Black se posicionó como el producto más vendido del período analizado.

Desde la perspectiva de adquisición de clientes, los tres canales de marketing mostraron un desempeño eficiente, generando más de tres unidades monetarias de ingresos por cada unidad invertida. Paid Search obtuvo el mejor resultado, con un ROAS de 3,48, seguido de Social (3,30) y Organic (3,27), lo que evidencia una distribución equilibrada del rendimiento entre canales.

No obstante, el análisis detallado de la rentabilidad identificó 4.528 ventas con utilidad negativa, equivalentes al 18,18 % de las transacciones. Aunque estas operaciones generaron pérdidas puntuales, el impacto no se distribuye de manera homogénea entre los productos. Se encontró que Laptop-Gaming-16GB es el único producto con rentabilidad neta negativa, registrando una pérdida acumulada de $93.621,66 y un margen de -8,63 %, además de presentar ventas con pérdida en el 54,12 % de sus transacciones. En contraste, productos como Jacket-Winter-M y Blender-XL-Red también registran ventas individuales con pérdida, pero mantienen una rentabilidad positiva en el agregado.

Hallazgo principal: La rentabilidad general del negocio es sólida; sin embargo, existe una oportunidad clara de mejora en la gestión de Laptop-Gaming-16GB, ya que concentra la principal fuente de pérdidas. Se recomienda revisar su estructura de costos, política de descuentos y estrategia de precios antes de incrementar la inversión en adquisición de clientes, ya que optimizar este producto tendría un impacto directo sobre la rentabilidad global del negocio.


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [85]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [86]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [87]:
#Revisar datos
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id_usuario          120000 non-null  object
 1   id_sesion           120000 non-null  object
 2   nombre_evento       120000 non-null  object
 3   timestamp_evento    120000 non-null  object
 4   pais                120000 non-null  object
 5   dispositivo         120000 non-null  object
 6   fuente_referencia   120000 non-null  object
 7   categoria_producto  120000 non-null  object
dtypes: object(8)
memory usage: 7.3+ MB


In [88]:
#Convertir timestamp_evento
events["timestamp_evento"] = pd.to_datetime(events["timestamp_evento"])
print(events["timestamp_evento"].dtypes)

datetime64[ns]


In [89]:
#Eventos
events["nombre_evento"].unique().tolist()

['first_visit',
 'add_to_cart',
 'add_payment_info',
 'purchase',
 'begin_checkout',
 'select_item']

In [90]:
#identifica cuántos usuarios únicos llegan a cada evento
funnel = (
    events.groupby("nombre_evento")["id_usuario"]
    .nunique()
    .reset_index()
    .rename(columns={"id_usuario":"usuarios"})
)

print(funnel)

      nombre_evento  usuarios
0  add_payment_info      6250
1       add_to_cart      7634
2    begin_checkout      7208
3       first_visit      7796
4          purchase      6240
5       select_item      7582


In [91]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [92]:
#investigar una posible anomalía en el funnel
query_usuarios_sin_select = '''
SELECT COUNT(DISTINCT id_usuario) AS usuarios_sin_select
FROM events
WHERE nombre_evento = 'add_to_cart'
  AND id_usuario NOT IN (
      SELECT id_usuario
      FROM events
      WHERE nombre_evento = 'select_item'
  )
'''

usuarios_sin_select_item = pd.read_sql(query_usuarios_sin_select, con=engine)
usuarios_sin_select_item

,usuarios_sin_select
0,401


📝 **Comentario**:Durante la construcción del funnel se observó que el evento add_to_cart registraba más usuarios únicos (7.634) que select_item (7.582), una situación que no coincide con el flujo de compra esperado. Para investigar esta diferencia, se realizó una subconsulta con el fin de identificar los usuarios que añadieron productos al carrito sin haber registrado previamente un evento select_item.

El análisis mostró que 401 usuarios realizaron esta acción, lo que sugiere la existencia de rutas alternativas de navegación dentro de la plataforma o posibles limitaciones en la captura de eventos. Por esta razón, el funnel se analizará considerando que no todos los usuarios siguen necesariamente la secuencia completa de eventos. No obstante, el orden definido continúa representando adecuadamente las principales etapas del proceso de conversión.

🎯 **Hallazgo**
La diferencia observada entre las etapas select_item y add_to_cart no corresponde a un error en los datos. Se identificó que 401 usuarios agregaron productos al carrito sin registrar previamente una selección de producto, lo que evidencia la existencia de recorridos alternativos dentro de la plataforma. Este comportamiento deberá tenerse en cuenta al interpretar las tasas de conversión entre ambas etapas y los resultados generales del funnel.

💡 Este hallazgo sugiere que el evento select_item no captura la totalidad de los recorridos de compra, por lo que no debe interpretarse como un paso obligatorio para todos los usuarios dentro del proceso de conversión.

In [93]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS total_usuarios,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden_funnel
    FROM events
    GROUP BY nombre_evento
),

conversiones AS (
    SELECT
        nombre_evento,
        total_usuarios,
        orden_funnel,
        LAG(total_usuarios) OVER (
            ORDER BY orden_funnel
        ) AS usuarios_anteriores
    FROM funnel
)

SELECT
    nombre_evento,
    usuarios_anteriores,
    total_usuarios,
    usuarios_anteriores - total_usuarios AS usuarios_perdidos,
    ROUND(total_usuarios * 100.0 / NULLIF(usuarios_anteriores, 0), 2) AS conversion_pct,
    ROUND(100.0 - (total_usuarios * 100.0 / NULLIF(usuarios_anteriores, 0)), 2) AS drop_off_pct
FROM conversiones
WHERE usuarios_anteriores IS NOT NULL
ORDER BY usuarios_perdidos DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)

conversion


,nombre_evento,usuarios_anteriores,total_usuarios,usuarios_perdidos,conversion_pct,drop_off_pct
0,add_payment_info,7208,6250,958,86.71,13.29
1,begin_checkout,7634,7208,426,94.42,5.58
2,select_item,7796,7582,214,97.26,2.74
3,purchase,6250,6240,10,99.84,0.16
4,add_to_cart,7582,7634,-52,100.69,-0.69


In [94]:
query_conversion_global = '''
WITH conteos AS (
    SELECT
        COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS usuarios_first_visit,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS usuarios_purchase
    FROM events)

SELECT
    usuarios_first_visit,
    usuarios_purchase,
    ROUND(usuarios_purchase * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_global_pct
FROM conteos;
'''

conversion_global = pd.read_sql(query_conversion_global, con=engine)
conversion_global

,usuarios_first_visit,usuarios_purchase,conversion_global_pct
0,7796,6240,80.04


📝 **Conclusión: Funnel de conversión**

Se construyó un funnel de conversión a partir de los eventos registrados en la plataforma, con el objetivo de identificar el comportamiento de los usuarios a lo largo del proceso de compra y detectar los principales puntos de abandono.

El análisis mostró que 7.796 usuarios realizaron una primera visita al sitio y 6.240 completaron una compra, lo que representa una conversión global del 80,04 %. Este resultado indica un nivel de conversión elevado, ya que aproximadamente 8 de cada 10 usuarios que ingresan a la plataforma finalizan una compra.

Al analizar las transiciones entre etapas, se observó que el principal punto de fuga se encuentra entre begin_checkout y add_payment_info, donde se pierden 958 usuarios, equivalentes al 13,29 % de quienes habían iniciado el proceso de compra. Este comportamiento sugiere que la introducción de los datos de pago constituye la principal barrera dentro del funnel y representa la mayor oportunidad de optimización.

La segunda pérdida más relevante se produce entre add_to_cart y begin_checkout, con 426 usuarios abandonando el proceso (5,58 %), mientras que la transición entre first_visit y select_item registra una pérdida relativamente baja de 214 usuarios (2,74 %).

Asimismo, se identificó que 401 usuarios añadieron productos al carrito sin registrar previamente el evento select_item, lo que evidencia la existencia de recorridos alternativos de navegación o posibles limitaciones en la captura de eventos. Por este motivo, la diferencia observada entre ambas etapas no se considera un error de los datos y debe interpretarse como un comportamiento válido dentro de la plataforma.

Finalmente, una vez que los usuarios ingresan su información de pago, la probabilidad de completar la compra es extremadamente alta: la conversión entre add_payment_info y purchase alcanza el 99,84 %, con una pérdida de apenas 10 usuarios.

🎯 **Hallazgo principal**

El funnel presenta un desempeño general muy sólido, con una conversión final superior al 80 %. Sin embargo, la etapa de registro de información de pago concentra la mayor pérdida de usuarios y constituye la principal oportunidad de mejora para incrementar las conversiones y las ventas del negocio. Optimizar esta fase podría generar un impacto directo sobre los ingresos sin necesidad de aumentar la inversión en adquisición de tráfico.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [95]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [96]:
#Revisar datos
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_usuario      8000 non-null   object
 1   fecha_registro  8000 non-null   object
 2   país            8000 non-null   object
 3   dispositivo     8000 non-null   object
 4   tipo_plan       8000 non-null   object
dtypes: object(5)
memory usage: 312.6+ KB


In [97]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(5)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [98]:
#Revisar datos
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             32000 non-null  object
 1   fecha_actividad        32000 non-null  object
 2   dias_despues_registro  32000 non-null  int64 
 3   activo                 32000 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1000.1+ KB


In [99]:
#Identificación de cohortes

query_mes_cohorte = '''
SELECT
    id_usuario,
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
FROM users;
'''

# Ejecutar la consulta
mes_cohorte = pd.read_sql(query_mes_cohorte, con=engine)
mes_cohorte

,id_usuario,mes_cohorte
0,user_0,2025-01-01 00:00:00+00:00
1,user_1,2025-01-01 00:00:00+00:00
2,user_2,2025-03-01 00:00:00+00:00
3,user_3,2025-03-01 00:00:00+00:00
4,user_4,2025-02-01 00:00:00+00:00
...,...,...
7995,user_7995,2025-05-01 00:00:00+00:00
7996,user_7996,2025-01-01 00:00:00+00:00
7997,user_7997,2025-04-01 00:00:00+00:00
7998,user_7998,2025-05-01 00:00:00+00:00


In [100]:
#Identificación total de usuarios por cohorte

query_total_usuarios = '''
SELECT
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM users
GROUP BY mes_cohorte
ORDER BY mes_cohorte;
'''

# Ejecutar la consulta
total_usuarios = pd.read_sql(query_total_usuarios, con=engine)
total_usuarios

,mes_cohorte,total_usuarios
0,2025-01-01 00:00:00+00:00,1627
1,2025-02-01 00:00:00+00:00,1444
2,2025-03-01 00:00:00+00:00,1636
3,2025-04-01 00:00:00+00:00,1606
4,2025-05-01 00:00:00+00:00,1687


In [101]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH retencion_porcentaje AS (
    SELECT
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS mes_cohorte,
        COUNT(DISTINCT u.id_usuario) AS total_usuarios,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w1,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w2,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w3

    FROM users AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
    GROUP BY mes_cohorte
)

SELECT
    mes_cohorte,
    total_usuarios,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / total_usuarios, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / total_usuarios, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / total_usuarios, 2) AS semana_3
FROM retencion_porcentaje
ORDER BY mes_cohorte;
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,mes_cohorte,total_usuarios,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


📝 **Comentario**

Se calculó la retención de usuarios por cohortes tomando como referencia el mes de registro de cada usuario y evaluando cuántos permanecieron activos durante las semanas 1, 2 y 3 posteriores a su incorporación a la plataforma. Para ello, se unieron las tablas users y user_activity, identificando los usuarios activos en los días 7, 14 y 21 desde su registro y calculando posteriormente el porcentaje de retención sobre el total de usuarios de cada cohorte.

Los resultados muestran un comportamiento estable entre las cohortes de enero a mayo de 2025. La retención en la semana 1 oscila entre 41,20 % y 42,84 %, mientras que la semana 2 se sitúa entre 40,07 % y 43,40 %. En la semana 3, la retención permanece cercana al 42 %, con valores entre 40,32 % y 43,98 %.

A diferencia de lo que ocurre habitualmente en muchos productos digitales, no se observa una caída pronunciada de la retención entre semanas. Esto sugiere que los usuarios que continúan activos después de registrarse tienden a mantener un nivel de interacción relativamente constante durante las primeras semanas de uso.

🎯 **Hallazgos**
La retención semanal se mantiene estable alrededor del 42 % en todas las cohortes analizadas.
La cohorte de febrero de 2025 presentó la mejor retención en la semana 3 (43,98 %).
La cohorte de mayo de 2025 registró la menor retención en la semana 2 (40,07 %), aunque la diferencia frente a las demás cohortes es reducida.
No se identifican cambios significativos entre cohortes, lo que sugiere un comportamiento consistente de los usuarios independientemente de la fecha de adquisición.
La estabilidad de las tasas de retención indica que el proceso de incorporación y la experiencia inicial del usuario presentan un desempeño homogéneo durante el período analizado.

📌 **Conclusión ejecutiva**

El análisis de cohortes muestra que aproximadamente 4 de cada 10 usuarios continúan activos durante las tres primeras semanas posteriores a su registro. Además, las tasas de retención se mantienen estables entre cohortes y entre semanas, sin evidenciar deterioros significativos en la permanencia de los usuarios. Estos resultados sugieren que la plataforma presenta una capacidad consistente para retener usuarios en el corto plazo, aunque existe una oportunidad de mejora en las estrategias de fidelización para incrementar la retención por encima del 42 % y maximizar el valor generado por los usuarios adquiridos.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [102]:
# Cargar datos
experiment_checkout_ui = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")

In [103]:
#Explorar tabla
experiment_checkout_ui.info()
experiment_checkout_ui.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [104]:
#Resumen estadístico
experiment_checkout_ui.describe()

,convirtio,duracion_sesion
count,10000.000000,10000.000000
mean,0.159900,159.862439
std,0.366532,81.074410
min,0.000000,20.010000
25%,0.000000,89.470000
50%,0.000000,159.725000
75%,0.000000,229.745000
max,1.000000,300.000000


In [105]:
#Registros duplicados
experiment_checkout_ui[experiment_checkout_ui.duplicated(keep=False)].sort_values(by="id_usuario").head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp


In [106]:
#Valores unicos
experiment_checkout_ui["variante"].unique()

array(['tratamiento', 'control'], dtype=object)

In [107]:
#Contar usuarios en cada grupo
experiment_checkout_ui["variante"].value_counts()

tratamiento    5035
control        4965
Name: variante, dtype: int64

In [ ]:
# Importar la función proportions_ztest para realizar una prueba Z de dos proporciones.
from statsmodels.stats.proportion import proportions_ztest


In [ ]:
# Resumir los resultados del experimento A/B.
# Para cada variante (control y tratamiento) se calcula:
# - conversiones: usuarios que completaron la compra.
# - usuarios: tamaño total de la muestra.
# - tasa_conversion: porcentaje de usuarios que realizaron una compra.
# Además, se extraen los valores necesarios para ejecutar la prueba Z
# de diferencia de proporciones.

resumen_ab = experiment_checkout_ui.groupby("variante")["convirtio"].agg(
    conversiones="sum",
    usuarios="count",
    tasa_conversion="mean"
)

display(resumen_ab)

# Insumos para el test estadístico
exitos = resumen_ab["conversiones"].tolist()
observaciones = resumen_ab["usuarios"].tolist()

print("Éxitos:", exitos)
print("Observaciones:", observaciones)

🎯 Hipótesis nula (H₀)

No existe una diferencia estadísticamente significativa en la tasa de conversión entre el grupo de control y el grupo de tratamiento. Cualquier diferencia observada se debe al azar.

🎯 Hipótesis alternativa (H₁)

Existe una diferencia estadísticamente significativa en la tasa de conversión entre el grupo de control y el grupo de tratamiento.


In [ ]:
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico Z: {z_stat}")
print(f"Valor p: {p_value}")
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipótesis nula: Existe una diferencia estadísticamente significativa "
        "en la tasa de conversión entre el grupo de control y el grupo de tratamiento")
else:
    print("No rechazamos la hipótesis nula: No existe evidencia suficiente para afirmar que "
        "la tasa de conversión difiere entre el grupo de control y el grupo de tratamiento")

In [ ]:
#Calcular el tamaño de muestra necesario
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Conversión observada en el grupo control
p1 = 0.1569

# Conversión observada en tratamiento
p2 = 0.1629

# Tamaño del efecto observado
effect_size = proportion_effectsize(p1, p2)

# Análisis de potencia
power_analysis = NormalIndPower()

muestra_necesaria = power_analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    ratio=1,
    alternative='two-sided'
)

print(f"Muestra necesaria por grupo: {muestra_necesaria:.0f}")

In [ ]:
#Calcular el efecto mínimo detectable con tu muestra actual
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

power_analysis = NormalIndPower()

n_actual = 5000

for mejora in [0.01, 0.015, 0.02]:
    
    p1 = 0.1569
    p2 = p1 + mejora
    
    effect_size = proportion_effectsize(p1, p2)
    
    potencia = power_analysis.power(
        effect_size=effect_size,
        nobs1=n_actual,
        alpha=0.05,
        ratio=1,
        alternative='two-sided'
    )
    
    print(
        f"Mejora de {mejora*100:.1f} pp -> Potencia: {potencia:.3f}"
    )

📝 **Conclusión final del experimento A/B**

Se comparó la tasa de conversión entre las variantes control y tratamiento mediante una prueba Z para diferencia de proporciones, utilizando un nivel de significancia de α = 0,05.

La tasa de conversión observada fue de 15,69 % en el grupo control (779 conversiones de 4.965 usuarios) y de 16,29 % en el grupo tratamiento (820 conversiones de 5.035 usuarios), lo que representa una mejora de aproximadamente 0,60 puntos porcentuales a favor de la nueva interfaz.

Sin embargo, la prueba estadística obtuvo un estadístico Z de -0,81 y un valor p de 0,416, superior al nivel de significancia establecido. Por lo tanto, no se rechaza la hipótesis nula, concluyendo que no existe evidencia estadística suficiente para afirmar que la modificación de la interfaz de checkout produjo un cambio significativo en la tasa de conversión.

No obstante, la ausencia de significancia estadística no implica necesariamente que el efecto sea inexistente. Para evaluar esta limitación, se realizó un análisis de potencia estadística, considerando una potencia objetivo del 80 % y un nivel de significancia del 5 %.

Los resultados indican que, para detectar una mejora similar a la observada en el experimento (0,6 puntos porcentuales), serían necesarios aproximadamente 58.570 usuarios por grupo, una cifra considerablemente superior a los cerca de 5.000 usuarios por grupo utilizados en este análisis. Adicionalmente, se observó que la muestra actual presenta una potencia de apenas 27,4 % para detectar mejoras de 1 punto porcentual, 52,6 % para mejoras de 1,5 puntos porcentuales y 76,5 % para mejoras de 2 puntos porcentuales, valores inferiores al nivel recomendado del 80 %.

🎯 **Hallazgo principal**

Aunque la variante de tratamiento mostró una tasa de conversión ligeramente superior a la del grupo control, la diferencia observada no fue estadísticamente significativa y la muestra disponible no parece tener suficiente tamaño para detectar efectos pequeños con alta confianza estadística. Por ello, no se recomienda implementar la nueva interfaz basándose únicamente en los resultados actuales.

📌 **Recomendación**

Se recomienda repetir el experimento con una muestra significativamente mayor o mantener la prueba activa durante un período más prolongado. Un tamaño de muestra más amplio permitiría detectar mejoras pequeñas pero potencialmente relevantes para el negocio, reduciendo el riesgo de descartar cambios que podrían generar un impacto positivo en la conversión. De esta forma, futuras decisiones sobre la implementación de la nueva interfaz podrán sustentarse en evidencia estadística más robusta y concluyente.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive